In [ ]:
# load_breast_cancer()는 유방암 예제 데이터와 정답/특성 이름을 불러옵니다.
# pd.DataFrame()은 배열 데이터를 행과 열이 있는 표로 만듭니다. columns는 열 이름입니다.
# df.head()는 표의 앞 5행을 확인합니다.

import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

cancer = load_breast_cancer()

df = pd.DataFrame(cancer.data, columns= cancer.feature_names)
df.head()

In [ ]:
# LogisticRegression()은 로지스틱 회귀 분류기를 만듭니다. solver는 학습 계산 방법입니다.
# KNeighborsClassifier()는 가까운 이웃을 참고하는 분류기입니다. n_neighbors=8은 이웃 8개를 사용합니다.
# VotingClassifier()는 여러 분류기의 예측을 합치는 앙상블입니다. soft는 각 클래스 확률을 평균냅니다.
# train_test_split()은 데이터를 학습/평가용으로 분리합니다. test_size=0.2는 20% 평가용입니다.
# fit()은 학습, predict()는 예측, accuracy_score()는 정답 일치 비율을 계산합니다.
# print()는 결과를 출력합니다. __class__.__name__은 객체의 클래스 이름을 가져옵니다.

lr_clf = LogisticRegression(solver= "liblinear")
knn_clf = KNeighborsClassifier(n_neighbors=8)

vo_clf = VotingClassifier(estimators=[("LR",lr_clf),("KNN", knn_clf)],voting="soft")

X_train,X_test,y_train,y_test = train_test_split(cancer.data , cancer.target, test_size= 0.2 , random_state= 156)

vo_clf.fit (X_train,y_train)
pred = vo_clf.predict(X_test)
print(accuracy_score(y_test,pred))

classifiers = [lr_clf, knn_clf]
for classifier in classifiers : 
    classifier.fit(X_train, y_train)
    pred = classifier.predict(X_test)
    print(classifier.__class__.__name__,accuracy_score(y_test, pred))



In [ ]:
# get_new_feature_name_df()는 중복된 특성 이름에 순번을 붙여 고유한 이름으로 바꾸는 함수입니다.
# groupby().cumcount()는 같은 이름 그룹 안에 0부터 번호를 매깁니다.
# pd.DataFrame()은 번호를 표로 만들고 reset_index()는 인덱스를 열로 옮깁니다.
# pd.merge()는 두 표를 합칩니다. how="outer"는 양쪽 행을 모두 보존합니다.
# apply()는 행/열마다 함수를 실행하고 axis=1은 행 단위입니다. lambda는 짧은 익명 함수입니다.
# str()은 숫자를 문자열로 바꿔 이름과 연결하게 합니다. drop()은 지정 열을 제거합니다.
# return은 만들어진 새 특성 이름 표를 호출한 곳에 돌려줍니다.

def get_new_feature_name_df(old_feature_name_df):
    feature_dup_df = pd.DataFrame(data = old_feature_name_df.groupby('column_name').cumcount(),
                                  columns = ['dup_cnt'])
    feature_dup_df = feature_dup_df.reset_index()
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how = 'outer')
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x: x[0] + '_' + str(x[1]) if x[1] > 0 else x[0], axis = 1)
    new_feature_name_df = new_feature_name_df.drop(['index'], axis = 1)
    return new_feature_name_df

In [ ]:
# csv.reader()는 CSV 내용을 행별로 읽고 next()는 첫 헤더 행을 가져옵니다.
# open()은 파일을 열며 encoding="utf-8"은 문자 인코딩을 지정합니다.
# pd.DataFrame()은 표 생성, range()는 순번 범위 생성, len()은 항목 개수 계산입니다.
# get_new_feature_name_df()는 앞서 만든 함수로 중복 특성 이름을 정리합니다.
# tolist()는 열을 파이썬 리스트로 바꿉니다. pd.read_csv()는 CSV를 표로 읽습니다.
# train_df[feature_name]은 특성 열을 고르고 train_df[["action"]]은 정답 열을 고릅니다.
# return은 학습/테스트 특성과 정답 네 가지를 반환합니다.

import csv
import pandas as pd

ACTION_MAP = {
    "WALKING": 1,
    "WALKING_UPSTAIRS": 2,
    "WALKING_DOWNSTAIRS": 3,
    "SITTING": 4,
    "STANDING": 5,
    "LAYING": 6,
}

def get_human_dataset():
    train_file = "./human_activity_train.csv"
    test_file = "./human_activity_test.csv"

    with open('./human_activity_train.csv', encoding = 'utf-8') as f:
        header = next(csv.reader(f))
    feature_name_org = header[:-2]

    feature_name_df = pd.DataFrame({
        "column_index": range(1, len(feature_name_org) + 1),
        "column_name": feature_name_org
    })

    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    feature_name = new_feature_name_df["column_name"].tolist()

    col_names = feature_name + ["subject", "action"]
    train_df = pd.read_csv(train_file, header=0, names=col_names)
    test_df = pd.read_csv(test_file, header=0, names=col_names)
    X_train = train_df[feature_name]
    X_test = test_df[feature_name]
    y_train = train_df[["action"]]
    y_test = test_df[["action"]]

    return X_train, X_test, y_train, y_test

In [ ]:
# get_human_dataset()은 앞에서 정의한 함수로 CSV를 읽고 X/y로 나눕니다.
# shape는 표의 (행 수, 열 수)를 담은 속성입니다. print()는 크기를 화면에 표시합니다.

X_train,X_test,y_train,y_test = get_human_dataset()
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# RandomForestClassifier()는 여러 결정 트리를 모아 예측하는 랜덤 포레스트입니다.
# warnings.filterwarnings()는 경고 표시 방식을 설정합니다.
# fit()은 학습, predict()는 테스트 예측, accuracy_score()는 정답 일치 비율 계산입니다.
# print()는 정확도를 출력합니다.

# forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

rf_clf = RandomForestClassifier(random_state = 0)
rf_clf.fit (X_train, y_train)

pred = rf_clf.predict(X_test)
accuracy = accuracy_score(y_test, pred)
print(accuracy)


In [ ]:
# GridSearchCV()는 지정된 설정 후보를 교차 검증하며 비교합니다.
# RandomForestClassifier()는 랜덤 포레스트 모델을 만듭니다. n_jobs=-1은 가능한 CPU 코어를 사용합니다.
# fit()은 후보 설정 탐색을 실행합니다. cv=2는 2부분 교차 검증입니다.
# best_params_는 가장 좋은 설정, best_score_는 그 설정의 평균 검증 점수입니다.
# print()는 두 값을 출력합니다.

from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators" : [100],
    "max_depth" : [6,8,10,12],
    "min_samples_leaf" : [ 8,12,18],
    "min_samples_split" : [8,16,20]
}

rf_clf = RandomForestClassifier(random_state= 0, n_jobs = -1)
grid_cv = GridSearchCV(rf_clf, param_grid = params , cv= 2 , n_jobs = -1)
grid_cv.fit(X_train,y_train)

print(grid_cv.best_params_)
print(grid_cv.best_score_)


In [ ]:
# RandomForestClassifier()는 랜덤 포레스트를 만듭니다. n_estimators는 트리 수입니다.
# max_depth는 트리 최대 깊이, min_samples_leaf는 잎의 최소 표본 수,
# min_samples_split은 분기하려는 노드의 최소 표본 수를 지정합니다.
# fit()은 학습하고 predict()는 테스트 정답을 예측합니다.
# accuracy_score()는 예측과 실제 정답이 일치한 비율을 구하고 print()는 출력합니다.

rf_clf1 = RandomForestClassifier(n_estimators= 300, max_depth = 10, min_samples_leaf= 8 , min_samples_split= 8, random_state= 0 ) # 랜덤 포레스트 안에 트리 300개
rf_clf1.fit(X_train,y_train)
pred = rf_clf1.predict(X_test)
print(accuracy_score(y_test, pred))



In [ ]:
# feature_importances_는 모델이 계산한 특성별 중요도입니다.
# pd.Series()는 중요도 값과 특성 이름을 연결한 시리즈를 만듭니다.
# sort_values()는 값 기준 정렬, ascending=False는 큰 값부터 정렬합니다. [:20]은 상위 20개 선택입니다.
# plt.Figure()는 그림 크기를 준비하고 plt.title()은 제목을 지정합니다.
# sns.barplot()은 막대그래프, plt.show()는 그래프 표시 함수입니다.

import matplotlib.pyplot as plt
import seaborn as sns

ftr_importances_values = rf_clf1.feature_importances_
ftr_importances = pd.Series(ftr_importances_values, index = X_train.columns)
ftr_top20 = ftr_importances.sort_values(ascending = False)[:20]

plt.Figure(figsize=(8,6)) 
plt.title("Feature Important top 20")
sns.barplot(x = ftr_top20, y= ftr_top20.index)
plt.show()
## 왜 무지개색이냐..?

In [ ]:
# GradientBoostingClassifier()는 이전 모델의 오류를 보완하며 트리를 차례로 학습하는 분류기입니다.
# time.time()은 현재 시각(초)을 반환해 실행 시간을 재는 데 씁니다.
# get_human_dataset()은 학습/테스트 데이터를 준비합니다.
# fit()은 학습, predict()는 예측, accuracy_score()는 정답 일치 비율 계산입니다.
# print()는 정확도와 시작 후 경과 시간을 출력합니다.

from sklearn.ensemble import GradientBoostingClassifier
import time
X_train, X_test, y_train, y_test = get_human_dataset()

start_time =time.time()

gb_clf = GradientBoostingClassifier(random_state= 0 )
gb_clf.fit(X_train,y_train)
gb_pred = gb_clf.predict(X_test)
print(accuracy_score(y_test, gb_pred))
print(time.time() - start_time)

In [ ]:
# 이 셀에는 실행 코드가 없습니다. 위 셀의 시간 측정 방식 등을 메모하는 자리입니다.

#  경사 하강법 = ?? 



In [ ]:
# GridSearchCV()는 설정 후보를 교차 검증하는 탐색기입니다.
# fit()은 후보 조합 탐색을 실행하고 verbose=1은 진행 상황을 출력합니다.
# best_params_는 최선 설정, best_score_는 최고 교차 검증 점수이며 print()로 표시합니다.

from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators" : [100, 500],
    "learning_rate" : [0.05, 0.1]
}
grid_cv = GridSearchCV(gb_clf, param_grid = params, cv = 2 , verbose = 1)
grid_cv.fit(X_train,y_train)

print(grid_cv.best_params_)
print(grid_cv.best_score_)

In [ ]:
# 이 셀은 비어 있습니다. 다음 셀에서 탐색 결과를 사용할 수 있습니다.



In [ ]:
# grid_cv.best_estimator는 GridSearchCV에서 선택된 최적 모델을 가리키려는 표현입니다.
# 일반적으로 scikit-learn 속성 이름은 best_estimator_처럼 끝에 밑줄이 붙습니다.
# 원본 코드는 변경하지 않았으므로 실행 여부/속성명은 그대로 두었습니다.

gb_pred = grid_cv.best_estimator